# 03 – Model Training

End-to-end training pipeline: data loading → feature engineering →
XGBoost and LSTM training → evaluation → comparison.


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.data.fetch_data import fetch_historical_data
from src.data.preprocess import feature_engineering_pipeline
from src.models.xgboost_model import DSEXGBoostPredictor
from src.models.lstm_model import DSEStockPredictor
from src.models.evaluate import evaluate_model, plot_predictions


## 1. Load and Prepare Data


In [ ]:
raw_df = fetch_historical_data('GP', '2020-01-01', '2024-01-01')
df = feature_engineering_pipeline(raw_df)

col_map = {c.lower(): c for c in df.columns}
close_col = col_map['close']
feature_cols = [c for c in df.columns if c != close_col]

print('Dataset shape:', df.shape)
print('Features:', len(feature_cols))


## 2. XGBoost Model


In [ ]:
xgb = DSEXGBoostPredictor(window_size=60)
X_train, y_train, X_val, y_val = xgb.prepare_data(df, feature_cols, close_col)

print(f'Train: {X_train.shape}  |  Val: {X_val.shape}')

xgb.train(X_train, y_train, X_val, y_val, n_estimators=300, early_stopping_rounds=20)
print('XGBoost training complete.')


In [ ]:
xgb_preds = xgb.predict(X_val)
y_val_orig = xgb.target_scaler.inverse_transform(y_val.reshape(-1, 1)).flatten()

xgb_metrics = evaluate_model(y_val_orig, xgb_preds)
print('XGBoost Metrics:', xgb_metrics)

plot_predictions(y_val_orig, xgb_preds, title='XGBoost – Actual vs Predicted')


## 3. LSTM Model


In [ ]:
lstm = DSEStockPredictor(window_size=60)
X_tr, y_tr, X_v, y_v = lstm.prepare_data(df, feature_cols, close_col)

print(f'Train: {X_tr.shape}  |  Val: {X_v.shape}')

history = lstm.train(X_tr, y_tr, X_v, y_v, epochs=50, batch_size=32)


In [ ]:
lstm_preds = lstm.predict(X_v)
y_v_orig = lstm.target_scaler.inverse_transform(y_v.reshape(-1, 1)).flatten()

lstm_metrics = evaluate_model(y_v_orig, lstm_preds)
print('LSTM Metrics:', lstm_metrics)

plot_predictions(y_v_orig, lstm_preds, title='LSTM – Actual vs Predicted')


## 4. Model Comparison


In [ ]:
import pandas as pd

comparison = pd.DataFrame(
    [xgb_metrics, lstm_metrics],
    index=['XGBoost', 'LSTM'],
)
print(comparison.to_string())

comparison[['rmse', 'mae']].plot(kind='bar', figsize=(8, 4))
plt.title('RMSE & MAE Comparison')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 5. Save Models


In [ ]:
xgb.save('../models/xgb_model')
print('XGBoost model saved.')

lstm.save('../models/lstm_model')
print('LSTM model saved.')
